# Baseline Linear Probe for Typosquat Detection
Trains a logistic regression probe on hidden states of Qwen-2.5-1.5B to separate clean from typosquatted commands.

In [ ]:
!pip install -q torch transformers scikit-learn matplotlib seaborn

In [ ]:
import json, random, re, numpy as np, pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Upload dataset
from google.colab import files
import zipfile, io

uploaded = files.upload()
fname = list(uploaded.keys())[0]

if fname.endswith('.zip'):
    with zipfile.ZipFile(io.BytesIO(uploaded[fname])) as zf:
        jsonl = [n for n in zf.namelist() if n.endswith('.jsonl')][0]
        zf.extract(jsonl)
        DATASET_PATH = jsonl
else:
    DATASET_PATH = fname
print(f"Using dataset: {DATASET_PATH}")

In [ ]:
df = pd.DataFrame([json.loads(line) for line in open(DATASET_PATH)])
df = df[df['is_adversarial'] == True].copy()
print(f"Loaded {len(df)} adversarial entries")

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
model.eval()

In [ ]:
def find_package_span(command: str, pkg: str):
    pattern = r'\b' + re.escape(pkg) + r'\b'
    match = re.search(pattern, command)
    return (match.start(), match.end()) if match else None

def char_to_token_span(tokenizer, text, cs, ce):
    enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=True)
    offs = enc['offset_mapping']
    ts = te = None
    for i, (s, e) in enumerate(offs):
        if s <= cs < e or s < ce <= e:
            if ts is None: ts = i
            te = i
    return (ts or len(offs)-1, te or len(offs)-1)

def extract_hidden_states(model, tokenizer, commands, packages, layers=(12,16), batch_size=8, max_length=512):
    model.eval()
    all_states = []
    for i in range(0, len(commands), batch_size):
        batch_cmds = commands[i:i+batch_size]
        batch_pkgs = packages[i:i+batch_size]
        inputs = tokenizer(batch_cmds, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            hidden = outputs.hidden_states
        for j, (cmd, pkg) in enumerate(zip(batch_cmds, batch_pkgs)):
            span = find_package_span(cmd, pkg)
            if span is None:
                states = [hidden[l][j, -1, :].float().cpu().numpy() for l in range(layers[0], min(layers[1]+1, len(hidden)))]
            else:
                ts, te = char_to_token_span(tokenizer, cmd, *span)
                states = []
                for l in range(layers[0], min(layers[1]+1, len(hidden))):
                    h = hidden[l][j, ts:te+1, :].mean(dim=0).float().cpu().numpy()
                    states.append(h)
            if states:
                combined = np.concatenate(states)
                combined = np.nan_to_num(combined, nan=0.0, posinf=0.0, neginf=0.0)
                all_states.append(combined)
    return np.array(all_states)

In [ ]:
texts, pkgs, labels = [], [], []
for _, row in df.iterrows():
    texts.append(row['clean_command']); pkgs.append(row['package_name']); labels.append(0)
    texts.append(row['typo_command']); pkgs.append(row['typo_package']); labels.append(1)

print("Extracting features...")
X = extract_hidden_states(model, tokenizer, texts, pkgs)
y = np.array(labels)
print(f"Feature matrix shape: {X.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)
probe = LogisticRegression(max_iter=1000, random_state=SEED)
probe.fit(X_train, y_train)
y_pred_proba = probe.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"Baseline Probe AUC: {auc:.4f}")